In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data/model")

In [2]:
train_data = pd.read_parquet(DATA_DIR / "train_fe.parquet").copy()
test_data = pd.read_parquet(DATA_DIR / "test_fe.parquet").copy()

In [3]:
print(f"Train raw shape: {train_data.shape}")
print(f"Test raw shape : {test_data.shape}")

Train raw shape: (830972, 28)
Test raw shape : (1113, 26)


In [4]:
display(train_data.head())
display(train_data.info())

,Store,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,...,IsWeekend,IsMonthStart,IsMonthEnd,CompetitionMonthsActive,Promo2WeeksActive,PromoIntervalActive,Lag_1,Lag_7,Rolling_Mean_7,Rolling_Std_7
0,1,3,5530.0,668,1,0,0,1,c,a,...,0,0,0,52.800000,NaN,0,0.0,NaN,NaN,NaN
1,1,4,4327.0,578,1,0,0,1,c,a,...,0,0,0,52.833333,NaN,0,5530.0,NaN,NaN,NaN
2,1,5,4486.0,619,1,0,0,1,c,a,...,0,0,0,52.866667,NaN,0,4327.0,NaN,NaN,NaN
3,1,6,4997.0,635,1,0,0,1,c,a,...,1,0,0,52.900000,NaN,0,4486.0,NaN,NaN,NaN
4,1,1,7176.0,785,1,1,0,1,c,a,...,0,0,0,52.966667,NaN,0,0.0,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830972 entries, 0 to 830971
Data columns (total 28 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Store                    830972 non-null  int64  
 1   DayOfWeek                830972 non-null  int32  
 2   Sales                    830972 non-null  float64
 3   Customers                830972 non-null  int64  
 4   Open                     830972 non-null  int64  
 5   Promo                    830972 non-null  int64  
 6   StateHoliday             830972 non-null  object 
 7   SchoolHoliday            830972 non-null  int64  
 8   StoreType                830972 non-null  object 
 9   Assortment               830972 non-null  object 
 10  CompetitionDistance      830972 non-null  float64
 11  Promo2                   830972 non-null  int64  
 12  CompetitionMissingFlag   830972 non-null  int64  
 13  LogSales                 830972 non-null  float64
 14  Year

None

## Phân loại các cột

In [5]:
TARGET_COL = "Sales"
ID_COL = "Store"
DATE_PARTS = ["Year", "Month", "Day"]
CAT_COLS = ["StateHoliday", "StoreType", "Assortment"]
LEAK_COLS = {TARGET_COL, "Customers", "LogSales", "Date"}
NA_ZERO_COLS = [
    "CompetitionMonthsActive",
    "Promo2WeeksActive",
    "Lag_1",
    "Lag_7",
    "Rolling_Mean_7",
    "Rolling_Std_7",
]

In [6]:
# Ensure optional columns exist for alignment
for col in [TARGET_COL, "Customers", "LogSales"]:
    if col not in test_data.columns:
        test_data[col] = np.nan

In [7]:

# Build timestamp from Year/Month/Day for ordering
for df in [train_data, test_data]:
    date_frame = df[DATE_PARTS].rename(columns={"Year": "year", "Month": "month", "Day": "day"})
    df["Date"] = pd.to_datetime(date_frame)

In [8]:
train_data["dataset"] = "train"
test_data["dataset"] = "test"
combined = pd.concat([train_data, test_data], ignore_index=True)

In [9]:
for col in CAT_COLS:
    combined[col] = combined[col].astype(str)

In [10]:
for col in NA_ZERO_COLS:
    if col in combined.columns:
        combined[col] = combined[col].fillna(0)

In [11]:
combined = pd.get_dummies(combined, columns=CAT_COLS, drop_first=True)
combined = combined.sort_values([ID_COL, "Date"]).reset_index(drop=True)

In [12]:
train_data = combined[combined["dataset"] == "train"].drop(columns=["dataset"]).reset_index(drop=True)
test_data = combined[combined["dataset"] == "test"].drop(columns=["dataset"]).reset_index(drop=True)

In [13]:
FEATURE_COLS = [col for col in train_data.columns if col not in LEAK_COLS and col != ID_COL]

print(f"Train shape (post-encoding): {train_data.shape}")
print(f"Test shape  (post-encoding): {test_data.shape}")
print(f"Feature count: {len(FEATURE_COLS)}")

Train shape (post-encoding): (830972, 34)
Test shape  (post-encoding): (1113, 34)
Feature count: 29


## Chia tập Validation theo thời gian 6 tuần

In [14]:
VAL_WEEKS = 6
seq_horizon = pd.Timedelta(weeks=VAL_WEEKS)
split_date = train_data["Date"].max() - seq_horizon

In [15]:
train_main = train_data[train_data["Date"] < split_date].copy()
val_main = train_data[train_data["Date"] >= split_date].copy()

In [16]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

In [17]:
train_main[FEATURE_COLS] = feature_scaler.fit_transform(train_main[FEATURE_COLS])
val_main[FEATURE_COLS] = feature_scaler.transform(val_main[FEATURE_COLS])
test_scaled = test_data.copy()
test_scaled[FEATURE_COLS] = feature_scaler.transform(test_scaled[FEATURE_COLS])

In [18]:
train_main[[TARGET_COL]] = target_scaler.fit_transform(train_main[[TARGET_COL]])
val_main[[TARGET_COL]] = target_scaler.transform(val_main[[TARGET_COL]])

In [19]:
print(f"Split date: {split_date.date()}")
print(f"Train rows: {len(train_main):,} | Val rows: {len(val_main):,}")

Split date: 2015-06-05
Train rows: 789,557 | Val rows: 41,415


## Build Sequences

In [20]:
SEQ_LEN = 30  # days
BATCH_SIZE = 256

def build_sequences(df: pd.DataFrame, feature_cols, target_col):
    sequences, targets = [], []
    for _, group in df.groupby(ID_COL):
        group = group.sort_values("Date")
        values = group[feature_cols + [target_col]].to_numpy()
        if len(values) <= SEQ_LEN:
            continue
        for start in range(len(values) - SEQ_LEN):
            seq_x = values[start:start + SEQ_LEN, :-1]
            seq_y = values[start + SEQ_LEN, -1]
            sequences.append(seq_x)
            targets.append(seq_y)
    return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32)

X_train, y_train = build_sequences(train_main, FEATURE_COLS, TARGET_COL)
X_val, y_val = build_sequences(val_main, FEATURE_COLS, TARGET_COL)

print(f"Train sequences: {X_train.shape}")
print(f"Val sequences  : {X_val.shape}")


Train sequences: (756107, 30, 29)
Val sequences  : (7977, 30, 29)


In [21]:
class SequenceDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

train_ds = SequenceDataset(X_train, y_train)
val_ds = SequenceDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"Batches -> train: {len(train_loader)}, val: {len(val_loader)}")


Batches -> train: 2954, val: 32


## Lưu Artifacts để dùng lại

In [ ]:
import pickle

# Tạo thư mục nếu chưa có
SAVE_DIR = DATA_DIR / "processed"
SAVE_DIR.mkdir(exist_ok=True)

# ============ 1. Lưu Sequences (numpy arrays) ============
np.save(SAVE_DIR / "X_train.npy", X_train)
np.save(SAVE_DIR / "y_train.npy", y_train)
np.save(SAVE_DIR / "X_val.npy", X_val)
np.save(SAVE_DIR / "y_val.npy", y_val)

print("Sequences đã lưu:")
print(f"   - X_train.npy: {X_train.shape}")
print(f"   - y_train.npy: {y_train.shape}")
print(f"   - X_val.npy: {X_val.shape}")
print(f"   - y_val.npy: {y_val.shape}")


✅ Sequences đã lưu:
   - X_train.npy: (756107, 30, 29)
   - y_train.npy: (756107,)
   - X_val.npy: (7977, 30, 29)
   - y_val.npy: (7977,)


In [ ]:
# ============ 2. Lưu Scalers (StandardScaler objects) ============
with open(SAVE_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(feature_scaler, f)

with open(SAVE_DIR / "target_scaler.pkl", "wb") as f:
    pickle.dump(target_scaler, f)

print("\nScalers đã lưu:")
print(f"   - feature_scaler.pkl")
print(f"   - target_scaler.pkl")



✅ Scalers đã lưu:
   - feature_scaler.pkl
   - target_scaler.pkl


In [ ]:
# ============ 3. Lưu Metadata (cấu hình) ============
metadata = {
    "FEATURE_COLS": FEATURE_COLS,
    "SEQ_LEN": SEQ_LEN,
    "BATCH_SIZE": BATCH_SIZE,
    "INPUT_DIM": len(FEATURE_COLS),
    "split_date": str(split_date),
    "TARGET_COL": TARGET_COL,
    "ID_COL": ID_COL,
}

with open(SAVE_DIR / "metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("\nMetadata đã lưu:")
print(f"   - metadata.pkl")
print(f"   - Số features: {metadata['INPUT_DIM']}")
print(f"   - Sequence length: {metadata['SEQ_LEN']}")
print(f"   - Batch size: {metadata['BATCH_SIZE']}")



✅ Metadata đã lưu:
   - metadata.pkl
   - Số features: 29
   - Sequence length: 30
   - Batch size: 256


In [ ]:
print("\nCấu trúc thư mục:")
for file in sorted(SAVE_DIR.glob("*")):
    size = file.stat().st_size / (1024**2)  # Convert to MB
    print(f"  - {file.name} ({size:.2f} MB)")



✅✅✅ TẤT CẢ ARTIFACTS ĐÃ LƯU VÀO: data\model\processed

Cấu trúc thư mục:
  - feature_scaler.pkl (0.00 MB)
  - metadata.pkl (0.00 MB)
  - target_scaler.pkl (0.00 MB)
  - X_train.npy (2509.36 MB)
  - X_val.npy (26.47 MB)
  - y_train.npy (2.88 MB)
  - y_val.npy (0.03 MB)
